# Clonal Dynamics (Cell + Nucleus)

For persisting, slow proliferator, and fast proliferator cells (dying
cells excluded - lineage continuity isn't the point there), finds the 2
`vem_1uM` clones (lineages, by `lin_id`) whose own cells are most
TIGHTLY CLUSTERED morphologically - lowest mean pairwise CAJAL GW
distance among themselves, on the REAL distance matrix, never the
UMAP/PHATE 2D projection (which distorts distances) - among clones with
enough cells in the embedded sample to make that mean meaningful. Paints
each one's own cells onto the embedding in its own color, against the
whole population in grey (one stacked, 2-panel figure per subclass, same
layout as Figure 1g of Shaffer et al., "Diverse clonal fates emerge upon
drug treatment of homogeneous cancer cells"), and separately crops each
clone's own MEDOID cell's actual shape - the cell with the lowest mean
real distance to every other cell in its own clone, not just whichever
cell happens to sit closest to the clone's average point on the 2D
plot, which the projection's own distortion could mislead.

This analysis (not `naive_vs_vem`) is the right place for this: its own
CAJAL run samples `vem_1uM` alone, more heavily per (frame, subclass)
than `naive_vs_vem`'s shared naive+vem embedding, so a given clone is
far more likely to have enough of its own cells actually embedded.

Clone SELECTION (`select_clones_by_subclass`) is the expensive part - it
scans every candidate lineage's own real distance submatrix - and
depends only on the real distance matrix, never on which embedding
method is being plotted, so it's run ONCE PER MODALITY and reused for
both UMAP and PHATE, rather than redone per embedding method.

See `shared/clonal_dynamics.py`'s own docstrings
(`select_clones_by_subclass`, `pick_tightest_clones`, `pick_clone_medoid`,
`plot_clones_on_embedding`, `plot_clone_shapes`, `export_clones_by_subclass`)
for exactly how a clone is selected, its medoid picked, and both plotted.

Plots both embeddings `main.py` computes (UMAP and PHATE) every time
this notebook runs, no manual switch needed.

Requires `main.py` to have been run first.

In [ ]:
import os
import sys

import pandas as pd

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/05_vem_full_timecourse/notebooks/), so the analysis dir -
# and config.py - is one level up, and the repo root - where shared/
# lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR/MASKS_DIR are relative to the
# analysis directory, not this notebook's own cwd - re-anchor them to an
# absolute path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)
config.MASKS_DIR        = os.path.join(ANALYSIS_DIR, config.MASKS_DIR)

from shared.load_data import load_full_dataset
from shared.cajal.embed import load_distance_matrix
from shared.load_data import parse_label
from shared.clonal_dynamics import select_clones_by_subclass, export_clones_by_subclass

## Plot: the 2 tightest clones per subclass, painted onto the embedding, plus each one's medoid shape

For each embedding method, one PNG and one SVG per subclass for the
embedding scatter, plus a SEPARATE PNG and SVG per subclass for the
clones' own medoid-cell shapes - written to
`output/plots/02_clonal_dynamics/` (SVGs in their own `svgs/`
subfolder there) - plus one
`{modality}_{embedding_method}_clone_summary.csv` recording exactly
which lin_ids were chosen and why (cell count, mean pairwise CAJAL
distance, and the medoid cell's own frame/pixel_id).

In [ ]:
N_CLONES = 2
MIN_EMBEDDED_CELLS = 10   # a clone needs at least this many embedded cells for its mean pairwise distance to be meaningful
SUBCLASSES_TO_PLOT = ['persisting', 'slow_prolif', 'fast_prolif']   # dying excluded - lineage continuity isn't the point there

# crop size (half-width in pixels) around each clone's own medoid cell -
# a SEPARATE, smaller value for 'nuc' than for 'cell', since a nucleus
# is much smaller than its whole cell - see shared.representative_shapes.
# export_representative_shapes for the full rationale. Both are a
# starting guess at "large enough for any {cell,nucleus} in this
# dataset" - crop_cell_shape prints a clear warning if a medoid cell's
# own shape doesn't fully fit, so bump the relevant one up if that happens.
HALF_WIDTH = {'cell': 150, 'nuc': 60}

NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '02_clonal_dynamics')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}

# main.py computes both of these for every modality - plot both every
# time this notebook runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def process_modality(modality):
    '''
    Load this modality's real CAJAL distance matrix once (both embedding
    methods are just two different 2D projections of that SAME distance
    matrix, so there's no need to reload it per method), then for each
    embedding method load its own 2D embedding and paint the 2 tightest
    clones per subclass onto it, plus their medoid shapes.

    Clone SELECTION (select_clones_by_subclass) only happens ONCE here,
    on whichever embedding method's own dataframe loads first - it
    never looks at dim_0/dim_1, so its result is identical regardless of
    embedding method, and redoing it per method would just repeat the
    same expensive per-lineage distance-matrix work for no reason.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        dfs: {embedding_method: the full dataset for this modality and
              embedding method} (in-memory only)
    '''
    dist_csv = os.path.join(config.CAJAL_OUTPUT_DIR, f'{modality}_qgw_{config.N_POINTS:03d}P_dists.csv')
    labels, dist_matrix = load_distance_matrix(dist_csv)
    label_to_index = {}
    for i, label in enumerate(labels):
        _, frame, pixel_id = parse_label(label)
        label_to_index[(frame, pixel_id)] = i

    selections = None
    dfs = {}
    for embedding_method in EMBEDDING_METHODS:
        df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality, embedding_method)
        print(f"[{modality}/{embedding_method}] {len(df)} cells across {df['subclass'].nunique()} subclasses")

        if selections is None:
            selections = select_clones_by_subclass(
                df, dist_matrix, label_to_index,
                n_clones=N_CLONES, min_embedded_cells=MIN_EMBEDDED_CELLS, subclasses=SUBCLASSES_TO_PLOT,
            )

        export_clones_by_subclass(
            df, selections, dist_matrix, label_to_index, config.MASKS_DIR, modality,
            NOTEBOOK_PLOTS_DIR, f'{modality}_{embedding_method}', embedding_method.upper(),
            title_prefix=f'{MODALITY_NAMES[modality]} morphology',
            n_clones=N_CLONES, half_width=HALF_WIDTH[modality],
        )
        dfs[embedding_method] = df

    return dfs

## Run: cell morphology

In [ ]:
cell_dfs = process_modality('cell')


## Run: nuclear morphology

In [ ]:
nuc_dfs = process_modality('nuc')
